# Structured Graph Yamada Dataset

This notebook systematically varies the parameters of every structured graph family in the Yamada catalog, excluding `bouquet` and `theta`, and writes a four-column CSV dataset with:

- `graph_name`
- `varying_params`
- `yamada`
- `yamada_sigma`

The sweep sizes are weighted by family complexity and target at least 500 rows overall.

In [1]:
import csv
import sys
from itertools import product
from pathlib import Path

import sympy as sp

repo_root = Path.cwd()
src_dir = repo_root / 'src'
if not src_dir.exists():
    raise RuntimeError('Run this notebook from the repository root so that ./src exists.')
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from knotted_graph.yamada import (
    GRAPH_FAMILY_CATALOG,
    compute_yamada_polynomial_recursive,
    laurent_y_to_sigma_polynomial,
)

Y = sp.Symbol('Y')
sigma = sp.Symbol('sigma')


In [4]:
FAMILY_TARGET_COUNTS = {
    'periodic_theta': 10,
    'sierpinski': 10,
    'complete_graph': 10,
    'complete_bipartite': 10,
    'complete_multipartite': 10,
    'fan': 10,
    'wheel': 10,
    'ladder': 10,
    'circular_ladder': 10,
    'grid': 10,
    'cylinder': 10,
    'hypercube': 10,
    'friendship': 10,
    'windmill': 10,
    'circulant': 10,
    'generalized_petersen': 10,
    'mobius_ladder': 10,
}
MIN_TOTAL_ROWS = 100


def take_first(sorted_candidates, count):
    return list(sorted_candidates[:count])


def periodic_theta_params(count):
    return [(s,) for s in range(1, count + 1)]


def sierpinski_params(count):
    candidates = [
        (n, t)
        for n in range(2, 14)
        for t in range(1, 8)
    ]
    candidates.sort(key=lambda item: (item[0] * item[1], item[1], item[0]))
    return take_first(candidates, count)


def complete_graph_params(count):
    return [(n,) for n in range(1, count + 1)]


def complete_bipartite_params(count):
    candidates = [
        (left_size, right_size)
        for left_size in range(1, 21)
        for right_size in range(left_size, 21)
    ]
    candidates.sort(key=lambda item: (sum(item), max(item), min(item)))
    return take_first(candidates, count)


def complete_multipartite_params(count):
    candidates = set()
    for num_parts in range(2, 6):
        for parts in product(range(1, 6), repeat=num_parts):
            ordered = tuple(sorted(parts, reverse=True))
            if sum(ordered) <= 10:
                candidates.add(ordered)
    ordered_candidates = sorted(
        candidates,
        key=lambda item: (sum(item), len(item), item),
    )
    return take_first(ordered_candidates, count)


def fan_params(count):
    return [(n_path_vertices,) for n_path_vertices in range(2, count + 2)]


def wheel_params(count):
    return [(n_rim_vertices,) for n_rim_vertices in range(3, count + 3)]


def ladder_params(count):
    return [(n_rungs,) for n_rungs in range(2, count + 2)]


def circular_ladder_params(count):
    return [(n_rungs,) for n_rungs in range(3, count + 3)]


def grid_params(count):
    candidates = [
        (rows, cols)
        for rows in range(1, 11)
        for cols in range(1, 13)
    ]
    candidates.sort(key=lambda item: (item[0] * item[1], item[0] + item[1], item))
    return take_first(candidates, count)


def cylinder_params(count):
    candidates = [
        (rows, cols)
        for rows in range(2, 11)
        for cols in range(2, 13)
    ]
    candidates.sort(key=lambda item: (item[0] * item[1], item[0] + item[1], item))
    return take_first(candidates, count)


def hypercube_params(count):
    return [(dimension,) for dimension in range(1, count + 1)]


def friendship_params(count):
    return [(n_triangles,) for n_triangles in range(1, count + 1)]


def windmill_params(count):
    candidates = [
        (clique_size, copies)
        for clique_size in range(3, 11)
        for copies in range(2, 21)
    ]
    candidates.sort(
        key=lambda item: (1 + item[1] * (item[0] - 1), item[0] + item[1], item)
    )
    return take_first(candidates, count)


def circulant_params(count):
    jump_sets = [
        (1,),
        (2,),
        (1, 2),
        (1, 3),
        (2, 3),
        (1, 4),
        (2, 4),
        (1, 2, 3),
    ]
    candidates = [
        (n_vertices, jumps)
        for n_vertices in range(4, 31)
        for jumps in jump_sets
        if max(jumps) < n_vertices
    ]
    candidates.sort(key=lambda item: (item[0], len(item[1]), item[1]))
    return take_first(candidates, count)


def generalized_petersen_params(count):
    candidates = [
        (n_vertices, step)
        for n_vertices in range(5, 31)
        for step in range(1, n_vertices // 2 + 1)
        if step < n_vertices / 2
    ]
    candidates.sort(key=lambda item: (item[0], item[1]))
    return take_first(candidates, count)


def mobius_ladder_params(count):
    return [(n_rungs,) for n_rungs in range(2, count + 2)]


PARAM_GENERATORS = {
    'periodic_theta': periodic_theta_params,
    'sierpinski': sierpinski_params,
    'complete_graph': complete_graph_params,
    'complete_bipartite': complete_bipartite_params,
    'complete_multipartite': complete_multipartite_params,
    'fan': fan_params,
    'wheel': wheel_params,
    'ladder': ladder_params,
    'circular_ladder': circular_ladder_params,
    'grid': grid_params,
    'cylinder': cylinder_params,
    'hypercube': hypercube_params,
    'friendship': friendship_params,
    'windmill': windmill_params,
    'circulant': circulant_params,
    'generalized_petersen': generalized_petersen_params,
    'mobius_ladder': mobius_ladder_params,
}


SYSTEMATIC_YAMADA_SWEEPS = {
    family_name: PARAM_GENERATORS[family_name](count)
    for family_name, count in FAMILY_TARGET_COUNTS.items()
}

excluded_families = {'bouquet', 'theta'}
catalog_families = set(GRAPH_FAMILY_CATALOG) - excluded_families
sweep_families = set(SYSTEMATIC_YAMADA_SWEEPS)
missing_families = sorted(catalog_families - sweep_families)
extra_families = sorted(sweep_families - catalog_families)
if missing_families or extra_families:
    raise ValueError(
        f'Sweep coverage mismatch. Missing={missing_families}, extra={extra_families}'
    )

planned_systematic_row_count = sum(
    len(args_list) for args_list in SYSTEMATIC_YAMADA_SWEEPS.values()
)
if planned_systematic_row_count < MIN_TOTAL_ROWS:
    raise ValueError(
        f'Planned row count {planned_systematic_row_count} is below {MIN_TOTAL_ROWS}'
    )


def compute_systematic_yamada_dataset(output_path):
    fieldnames = ['graph_name', 'varying_params', 'yamada', 'yamada_sigma']
    rows = []
    print(f'Planned total rows: {planned_systematic_row_count}')
    print(f"{'graph_name':24s} {'varying_params':20s} yamada")
    print('-' * 120)

    with output_path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()

        for family_name, args_list in SYSTEMATIC_YAMADA_SWEEPS.items():
            builder = GRAPH_FAMILY_CATALOG[family_name].builder
            print(f'\n[{family_name}] {len(args_list)} cases')
            for args in args_list:
                G = builder(*args)
                yamada_expr = sp.expand(compute_yamada_polynomial_recursive(G, Y))
                yamada_sigma_expr = laurent_y_to_sigma_polynomial(
                    yamada_expr,
                    Y,
                    sigma,
                ).as_expr()
                row = {
                    'graph_name': family_name,
                    'varying_params': repr(tuple(args)),
                    'yamada': sp.sstr(yamada_expr),
                    'yamada_sigma': sp.sstr(yamada_sigma_expr),
                }
                writer.writerow(row)
                handle.flush()
                rows.append(row)
                print(
                    f"{row['graph_name']:24s} {row['varying_params']:20s} {row['yamada']}"
                )

    return rows


systematic_yamada_dataset_path = repo_root / 'structured_graph_yamada_dataset.csv'
systematic_yamada_dataset = compute_systematic_yamada_dataset(
    systematic_yamada_dataset_path
)
print(f'\nSaved {len(systematic_yamada_dataset)} rows to {systematic_yamada_dataset_path}')
systematic_yamada_dataset[:5]


Planned total rows: 170
graph_name               varying_params       yamada
------------------------------------------------------------------------------------------------------------------------

[periodic_theta] 10 cases
periodic_theta           (1,)                 0
periodic_theta           (2,)                 Y**3 + 2*Y**2 + 5*Y + 5 + 5/Y + 2/Y**2 + Y**(-3)
periodic_theta           (3,)                 Y**5 + Y**4 + 7*Y**3 + 5*Y**2 + 15*Y + 8 + 15/Y + 5/Y**2 + 7/Y**3 + Y**(-4) + Y**(-5)
periodic_theta           (4,)                 Y**7 + 2*Y**6 + 13*Y**5 + 18*Y**4 + 60*Y**3 + 64*Y**2 + 125*Y + 97 + 125/Y + 64/Y**2 + 60/Y**3 + 18/Y**4 + 13/Y**5 + 2/Y**6 + Y**(-7)
periodic_theta           (5,)                 Y**9 + 3*Y**8 + 18*Y**7 + 36*Y**6 + 118*Y**5 + 167*Y**4 + 371*Y**3 + 396*Y**2 + 638*Y + 524 + 638/Y + 396/Y**2 + 371/Y**3 + 167/Y**4 + 118/Y**5 + 36/Y**6 + 18/Y**7 + 3/Y**8 + Y**(-9)
periodic_theta           (6,)                 Y**11 + 4*Y**10 + 24*Y**9 + 64*Y**8 + 214*Y**